In [2]:
import torch
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%%capture
!pip install transformers peft accelerate bitsandbytes datasets

print("Ready")

In [5]:
import torch
import json
import numpy as np
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from datasets import load_dataset

print("Imports done")

Imports done


In [6]:
# load base model in 4-bit (same as training)
model_name = "mistralai/Mistral-7B-v0.1"
ADAPTER_PATH = '/content/drive/MyDrive/adaptive-lora/checkpoints/adaptive'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # LEFT padding for generation

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

# load your trained LoRA adapter on top of base model
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()  # evaluation mode — turns off dropout

print("Model loaded with trained adapter")
print(f"Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded with trained adapter
Memory: 4.14 GB


In [ ]:
dataset = load_dataset("fancyzhx/ag_news")
label_names = ['World', 'Sports', 'Business', 'Sci/Tech']

def format_for_evaluation(example):
    # note: NO answer at the end — model must predict it
    text = f"""Classify the following news article into one of these categories: World, Sports, Business, Sci/Tech.

Article: {example['text']}

Category:"""
    return {"prompt": text, "label": example['label']}

test_data = dataset['test'].shuffle(seed=42).select(range(1000))
test_formatted = test_data.map(format_for_evaluation)

print(f"Test examples ready: {len(test_formatted)}")
print("\nExample prompt (last 200 chars):")
print(test_formatted[0]['prompt'][-200:])

In [8]:
correct = 0
total = 0
results = []

for example in tqdm(test_formatted):
    prompt = example['prompt']
    true_label = example['label']
    true_label_name = label_names[true_label]

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256,
        padding=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,       # ← CHANGED from 3 to 10
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # ← CHANGED: more robust label matching
    predicted_label = None
    generated_lower = generated.lower().strip()

    label_checks = {
        'World': ['world'],
        'Sports': ['sport'],
        'Business': ['business', 'busin'],
        'Sci/Tech': ['sci/tech', 'sci/t', 'sci-tech', 'science', 'tech']
    }

    for name, variants in label_checks.items():
        for variant in variants:
            if variant in generated_lower:
                predicted_label = name
                break
        if predicted_label:
            break

    is_correct = (predicted_label == true_label_name)
    if is_correct:
        correct += 1
    total += 1

    results.append({
        'true': true_label_name,
        'predicted': predicted_label,
        'generated': generated,
        'correct': is_correct
    })

accuracy = correct / total
print(f"\n=== ADAPTIVE RESULTS ===")
print(f"Correct: {correct}/{total}")
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

  0%|          | 0/1000 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
100%|██████████| 1000/1000 [11:14<00:00,  1.48it/s]


=== ADAPTIVE RESULTS ===
Correct: 910/1000
Accuracy: 0.9100 (91.00%)


In [9]:
# save accuracy
adaptive_results = {
    'accuracy': accuracy,
    'correct': correct,
    'total': total,
    'method': 'adaptive_uniform_r8'
}

with open('/content/drive/MyDrive/adaptive-lora/results/tables/adaptive_results.json', 'w') as f:
    json.dump(adaptive_results, f, indent=2)

# save detailed per-example results
with open('/content/drive/MyDrive/adaptive-lora/results/tables/adaptive_detailed.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved")
print(f"\nFinal Accuracy: {accuracy*100:.2f}%")

# quick breakdown by category
print("\nPer-category breakdown:")
from collections import defaultdict
category_correct = defaultdict(int)
category_total = defaultdict(int)

for r in results:
    category_total[r['true']] += 1
    if r['correct']:
        category_correct[r['true']] += 1

for cat in label_names:
    cat_acc = category_correct[cat] / category_total[cat]
    print(f"  {cat:10s}: {cat_acc*100:.1f}% ({category_correct[cat]}/{category_total[cat]})")

Results saved

Final Accuracy: 91.00%

Per-category breakdown:
  World     : 88.3% (235/266)
  Sports    : 99.6% (245/246)
  Business  : 81.3% (200/246)
  Sci/Tech  : 95.0% (230/242)
